# E2 linear probe — Kỳ hỗ trợ Trinh
Uses group trainer unchanged. No fabricated metrics. Run GPU preflight before training.

In [ ]:
# Nguyen Ngoc Ky supporting Trinh; isolated checkout, original E3/E5 preserved.
import os, sys, subprocess, base64, json
from pathlib import Path
import torch
print('CURRENT_GPU', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE', flush=True)
# Synthetic software checks may run on CPU; real E2 training is GPU-only.
ROOT=Path('/content/vit-oxford-pets-e2-support')
if not ROOT.exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/duyenphung206-sudo/vit-oxford-pets',str(ROOT)],check=True)
    subprocess.run(['git','-C',str(ROOT),'switch','-c','ky/support-e2-linear-probe','3359b2bdd2a081faa20f9ff83a93c2ccbac12071'],check=True)
os.chdir(ROOT)
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()=='ky/support-e2-linear-probe'
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()=='3359b2bdd2a081faa20f9ff83a93c2ccbac12071'
FILES={'configs/e2_vit_linear_probe.yaml': 'YmFzZTogZTRfdml0X2Z1bGxfZnQueWFtbApleHBlcmltZW50OiBlMl92aXRfbGluZWFyX3Byb2JlCm1vZGVsOgogIHN0cmF0ZWd5OiBsaW5lYXJfcHJvYmUKICB1bmZyZWV6ZV9sYXN0X2s6IDANCg==', 'src/ky_e2_runner.py': 'IiIiRTIgZXhlY3V0aW9uIGhlbHBlciBieSBOZ3V5ZW4gTmdvYyBLeSBzdXBwb3J0aW5nIFRyaW5oOyB1c2VzIHVuY2hhbmdlZCBzcmMudHJhaW4uCgpDb2xhYi1vbmx5LCBHUFUtb25seS4gRXBvY2ggbG9ncyBhbmQgYmVzdCBjaGVja3BvaW50IGFyZSB3cml0dGVuIGRpcmVjdGx5IHRvIERyaXZlLgpJbmNvbXBsZXRlIGVhcmxpZXIgcnVucyBhcmUgcHJlc2VydmVkOyB0aGV5IGFyZSBuZXZlciByZXN1bWVkIGZyb20gd2VpZ2h0cyBhbG9uZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYmFzZTY0CmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGlvCmltcG9ydCBqc29uCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKaW1wb3J0IHppcGZpbGUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgdG9yY2gKCmZyb20gc3JjLnN1bW1hcml6ZSBpbXBvcnQgcGxvdF9jdXJ2ZXMKZnJvbSBzcmMudXRpbHMgaW1wb3J0IGxvYWRfY29uZmlnCgoKZGVmIG1haW4oKToKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQkxPQ0tFRDogRTIgcmVxdWlyZXMgZnJlZSBDb2xhYiBHUFUsIG5vIENQVSB0cmFpbmluZyIpCiAgICByb290ID0gUGF0aCgiL2NvbnRlbnQvdml0LW94Zm9yZC1wZXRzLWUyLXN1cHBvcnQiKQogICAgaWYgUGF0aC5jd2QoKSAhPSByb290IG9yIHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KFsiZ2l0IiwgImJyYW5jaCIsICItLXNob3ctY3VycmVudCJdLCB0ZXh0PVRydWUpLnN0cmlwKCkgPT0gIm1haW4iOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiVXNlIGlzb2xhdGVkIEUyIHN1cHBvcnQgY2hlY2tvdXQsIG5vdCBtYWluIikKICAgIHN0b3JlID0gUGF0aCgiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS92aXQtb3hmb3JkLXBldHMta3kvZTItc3VwcG9ydCIpCiAgICBpZiBub3QgUGF0aCgiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZSIpLmlzX2RpcigpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTW91bnQgYXV0aG9yaXplZCBEcml2ZSBiZWZvcmUgdHJhaW5pbmciKQogICAgcnVuID0gc3RvcmUgLyB0aW1lLnN0cmZ0aW1lKCIlWSVtJWQtJUglTSVTIikKICAgIHJ1bi5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPUZhbHNlKQogICAgcmVzdWx0cyA9IHJ1biAvICJyZXN1bHRzL2UyX3ZpdF9saW5lYXJfcHJvYmUvc2VlZDQyIgogICAgY2hlY2twb2ludHMgPSBydW4gLyAiY2hlY2twb2ludHMvZTJfdml0X2xpbmVhcl9wcm9iZS9zZWVkNDIiCiAgICBjZmcgPSBsb2FkX2NvbmZpZygiY29uZmlncy9lMl92aXRfbGluZWFyX3Byb2JlLnlhbWwiKQogICAgYXNzZXJ0IGNmZ1sibW9kZWwiXVsic3RyYXRlZ3kiXSA9PSAibGluZWFyX3Byb2JlIiBhbmQgY2ZnWyJvcHRpbWl6ZXIiXVsibHIiXSA9PSAwLjAwMwogICAgYXNzZXJ0IGNmZ1siZGF0YSJdWyJiYXRjaF9zaXplIl0gPT0gNjQgYW5kIGNmZ1sidHJhaW4iXVsic2VlZCJdID09IDQyIGFuZCBjZmdbInRyYWluIl1bImVwb2NocyJdID09IDEwCiAgICBtZXRhZGF0YSA9IHsKICAgICAgICAiYXV0aG9yIjogIk5ndXllbiBOZ29jIEt5LCBzdXBwb3J0aW5nIFRyaW5oOyBub3QgYXR0cmlidXRlZCBhcyBUcmluaCBpbXBsZW1lbnRhdGlvbiIsCiAgICAgICAgInNvdXJjZV9jb21taXQiOiBzdWJwcm9jZXNzLmNoZWNrX291dHB1dChbImdpdCIsICJyZXYtcGFyc2UiLCAiSEVBRCJdLCB0ZXh0PVRydWUpLnN0cmlwKCksCiAgICAgICAgImdwdSI6IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLAogICAgICAgICJ0cmFpbmluZ19lbmdpbmUiOiAidW5jaGFuZ2VkIHNyYy50cmFpbiIsCiAgICAgICAgInJ1biI6IHN0cihydW4pLAogICAgICAgICJzdGF0dXMiOiAiUlVOTklORyIsCiAgICAgICAgInRlc3RfZXZhbHVhdGlvbiI6ICJOT1RfUlVOOyB2YWxpZGF0aW9uIG9ubHkiLAogICAgICAgICJoZWxwZXJfc2hhMjU2IjogaGFzaGxpYi5zaGEyNTYoUGF0aChfX2ZpbGVfXykucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAiY29uZmlnX3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KFBhdGgoImNvbmZpZ3MvZTJfdml0X2xpbmVhcl9wcm9iZS55YW1sIikucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgIH0KICAgIChydW4gLyAicnVuX21ldGFkYXRhLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWV0YWRhdGEsIGluZGVudD0yKSkKICAgIHByaW50KCJFMl9QRVJTSVNURU5UX1JVTiIsIHN0cihydW4pLCBmbHVzaD1UcnVlKQogICAgc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgWwogICAgICAgICAgICBzeXMuZXhlY3V0YWJsZSwKICAgICAgICAgICAgIi1tIiwKICAgICAgICAgICAgInNyYy50cmFpbiIsCiAgICAgICAgICAgICItLWNvbmZpZyIsCiAgICAgICAgICAgICJjb25maWdzL2UyX3ZpdF9saW5lYXJfcHJvYmUueWFtbCIsCiAgICAgICAgICAgICItLXNlZWQiLAogICAgICAgICAgICAiNDIiLAogICAgICAgICAgICAiLS1vdXRwdXQtZGlyIiwKICAgICAgICAgICAgc3RyKHJlc3VsdHMpLAogICAgICAgICAgICAiLS1jaGVja3BvaW50LWRpciIsCiAgICAgICAgICAgIHN0cihjaGVja3BvaW50cyksCiAgICAgICAgXSwKICAgICAgICBjaGVjaz1UcnVlLAogICAgKQogICAgbWV0cmljcyA9IGpzb24ubG9hZHMoKHJlc3VsdHMgLyAibWV0cmljcy5qc29uIikucmVhZF90ZXh0KCkpCiAgICByb3dzID0gbGlzdChjc3YuRGljdFJlYWRlcigocmVzdWx0cyAvICJsb2cuY3N2Iikub3BlbigpKSkKICAgIGFzc2VydCBtZXRyaWNzWyJmYWtlX2RhdGEiXSBpcyBGYWxzZSBhbmQgbWV0cmljc1sicGFyYW1ldGVyc190cmFpbmFibGUiXSA9PSAyODQ1MwogICAgYXNzZXJ0IGxlbihyb3dzKSA9PSBtZXRyaWNzWyJlcG9jaHNfcnVuIl0gYW5kIG1ldHJpY3NbInNlZWQiXSA9PSA0MgogICAgc2VsZWN0ZWQgPSByb3dzW21ldHJpY3NbImJlc3RfZXBvY2giXSAtIDFdCiAgICBhc3NlcnQgZmxvYXQoc2VsZWN0ZWRbInZhbF9sb3NzIl0pIDw9IG1pbihmbG9hdChyWyJ2YWxfbG9zcyJdKSBmb3IgciBpbiByb3dzKSArIDFlLTQKICAgIGFzc2VydCBhYnMoZmxvYXQoc2VsZWN0ZWRbInZhbF9hY2MiXSkgLSBtZXRyaWNzWyJiZXN0X3ZhbF9hY2MiXSkgPCA1LjFlLTUKICAgIGFzc2VydCBhYnMoZmxvYXQoc2VsZWN0ZWRbInZhbF9sb3NzIl0pIC0gbWV0cmljc1siYmVzdF92YWxfbG9zcyJdKSA8IDUuMWUtNQogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKGNoZWNrcG9pbnRzIC8gImJlc3QucHQiKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpCiAgICBtZXRhZGF0YS51cGRhdGUoCiAgICAgICAgc3RhdHVzPSJET05FIiwKICAgICAgICBjaGVja3BvaW50X3NoYTI1Nj1kaWdlc3QsCiAgICAgICAgY2hlY2twb2ludF9ieXRlcz0oY2hlY2twb2ludHMgLyAiYmVzdC5wdCIpLnN0YXQoKS5zdF9zaXplLAogICAgICAgIGVwb2Noc19ydW49bGVuKHJvd3MpLAogICAgICAgIGVsYXBzZWRfZXBvY2hfdGltZV9zPXN1bShmbG9hdChyWyJlcG9jaF90aW1lX3MiXSkgZm9yIHIgaW4gcm93cyksCiAgICAgICAgZ3B1X21lbW9yeV91bml0PSJNaUIgKHRyYWluZXIgZmllbGQgaXMgbmFtZWQgcGVha19ncHVfbWVtb3J5X21iKSIsCiAgICApCiAgICAocnVuIC8gInJ1bl9tZXRhZGF0YS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKG1ldGFkYXRhLCBpbmRlbnQ9MikpCiAgICBmaWd1cmUgPSBydW4gLyAiZTJfY3VydmVzLnBuZyIKICAgIHBsb3RfY3VydmVzKHJlc3VsdHMucGFyZW50LCBmaWd1cmUpCiAgICBwYXlsb2FkID0gaW8uQnl0ZXNJTygpCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShwYXlsb2FkLCAidyIsIHppcGZpbGUuWklQX0RFRkxBVEVEKSBhcyBidW5kbGU6CiAgICAgICAgZm9yIG5hbWUgaW4gWyJjb25maWcueWFtbCIsICJsb2cuY3N2IiwgIm1ldHJpY3MuanNvbiJdOgogICAgICAgICAgICBidW5kbGUud3JpdGUocmVzdWx0cyAvIG5hbWUsICJyZXN1bHRzL2UyX3ZpdF9saW5lYXJfcHJvYmUvc2VlZDQyLyIgKyBuYW1lKQogICAgICAgIGJ1bmRsZS53cml0ZShydW4gLyAicnVuX21ldGFkYXRhLmpzb24iLCAiZG9jcy9LWV9FMl9SVU5fTUFOSUZFU1QuanNvbiIpCiAgICBwcmludCgiRTJfTUVUUklDUyIsIGpzb24uZHVtcHMobWV0cmljcyksIGZsdXNoPVRydWUpCiAgICBwcmludCgiRTJfTUFOSUZFU1QiLCBqc29uLmR1bXBzKG1ldGFkYXRhKSwgZmx1c2g9VHJ1ZSkKICAgIHByaW50KCJFMl9URVhUX1pJUF9TSEEyNTYiLCBoYXNobGliLnNoYTI1NihwYXlsb2FkLmdldHZhbHVlKCkpLmhleGRpZ2VzdCgpLCBmbHVzaD1UcnVlKQogICAgcHJpbnQoIkUyX1RFWFRfWklQX0JBU0U2NCIsIGJhc2U2NC5iNjRlbmNvZGUocGF5bG9hZC5nZXR2YWx1ZSgpKS5kZWNvZGUoKSwgZmx1c2g9VHJ1ZSkKICAgIHByaW50KCJFMl9ET05FIiwgZmx1c2g9VHJ1ZSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==', 'src/ky_e2_checkpoint_audit.py': 'IiIiQXVkaXQgYSByZWFsIEUyIGNoZWNrcG9pbnQgYWdhaW5zdCBvcmlnaW5hbCBwcmV0cmFpbmVkIGJhY2tib25lIGFuZCB2YWxpZGF0aW9uIG1ldHJpY3MuIiIiDQppbXBvcnQgYXJncGFyc2UNCmltcG9ydCBqc29uDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KaW1wb3J0IHRvcmNoDQpmcm9tIHRvcmNodmlzaW9uLm1vZGVscyBpbXBvcnQgVmlUX0JfMTZfV2VpZ2h0cywgdml0X2JfMTYNCg0KZnJvbSBzcmMuZGF0YSBpbXBvcnQgYnVpbGRfZGF0YWxvYWRlcnMNCmZyb20gc3JjLmVuZ2luZSBpbXBvcnQgZXZhbHVhdGUNCmZyb20gc3JjLm1vZGVscy5mYWN0b3J5IGltcG9ydCBidWlsZF9tb2RlbA0KZnJvbSBzcmMudXRpbHMgaW1wb3J0IGNvdW50X3BhcmFtZXRlcnMsIGxvYWRfY29uZmlnDQoNCg0KZGVmIGF1ZGl0KHJ1bl9wYXRoLCBvdXRwdXQpOg0KICAgIHJ1biA9IFBhdGgocnVuX3BhdGgpDQogICAgbWFuaWZlc3QgPSBqc29uLmxvYWRzKChydW4vJ3J1bl9tZXRhZGF0YS5qc29uJykucmVhZF90ZXh0KCkpDQogICAgYXNzZXJ0IG1hbmlmZXN0WydzdGF0dXMnXSA9PSAnRE9ORScNCiAgICByZXN1bHRzID0gcnVuLydyZXN1bHRzL2UyX3ZpdF9saW5lYXJfcHJvYmUvc2VlZDQyJw0KICAgIGNmZyA9IGxvYWRfY29uZmlnKHJlc3VsdHMvJ2NvbmZpZy55YW1sJykNCiAgICBtZXRyaWNzID0ganNvbi5sb2FkcygocmVzdWx0cy8nbWV0cmljcy5qc29uJykucmVhZF90ZXh0KCkpDQogICAgc3RhdGUgPSB0b3JjaC5sb2FkKHJ1bi8nY2hlY2twb2ludHMvZTJfdml0X2xpbmVhcl9wcm9iZS9zZWVkNDIvYmVzdC5wdCcsDQogICAgICAgICAgICAgICAgICAgICAgIG1hcF9sb2NhdGlvbj0nY3B1Jywgd2VpZ2h0c19vbmx5PVRydWUpDQogICAgIyBMb2FkIHRocm91Z2ggdGhlIG9mZmljaWFsIG1vZGVsIHRvIG5vcm1hbGl6ZSBsZWdhY3kgTUxQIGNoZWNrcG9pbnQga2V5IG5hbWVzLg0KICAgIHJlZmVyZW5jZSA9IHZpdF9iXzE2KHdlaWdodHM9VmlUX0JfMTZfV2VpZ2h0cy5JTUFHRU5FVDFLX1YxKQ0KICAgIHByZXRyYWluZWQgPSByZWZlcmVuY2Uuc3RhdGVfZGljdCgpDQogICAgYmFja2JvbmVfa2V5cyA9IFtuYW1lIGZvciBuYW1lIGluIHN0YXRlIGlmIG5vdCBuYW1lLnN0YXJ0c3dpdGgoJ2hlYWRzLicpXQ0KICAgIGFzc2VydCBiYWNrYm9uZV9rZXlzIGFuZCBhbGwodG9yY2guZXF1YWwoc3RhdGVbbmFtZV0sIHByZXRyYWluZWRbbmFtZV0pIGZvciBuYW1lIGluIGJhY2tib25lX2tleXMpDQogICAgYXNzZXJ0IHN0YXRlWydoZWFkcy5oZWFkLndlaWdodCddLnNoYXBlID09ICgzNywgNzY4KQ0KICAgIGRlbCBwcmV0cmFpbmVkLCByZWZlcmVuY2UNCiAgICBtb2RlbF9jZmcgPSBkaWN0KGNmZ1snbW9kZWwnXSwgcHJldHJhaW5lZD1GYWxzZSkNCiAgICBtb2RlbCA9IGJ1aWxkX21vZGVsKG1vZGVsX2NmZykNCiAgICBsb2FkZWQgPSBtb2RlbC5sb2FkX3N0YXRlX2RpY3Qoc3RhdGUsIHN0cmljdD1UcnVlKQ0KICAgIGFzc2VydCBjb3VudF9wYXJhbWV0ZXJzKG1vZGVsKVsndHJhaW5hYmxlJ10gPT0gMjg0NTMNCiAgICBpZiBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdHUFUgdmFsaWRhdGlvbiByZXF1aXJlcyB0aGUgYXV0aG9yaXplZCBUNCBydW50aW1lJykNCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoJ2N1ZGEnKQ0KICAgIG1vZGVsLnRvKGRldmljZSkuZXZhbCgpDQogICAgbG9hZGVycyA9IGJ1aWxkX2RhdGFsb2FkZXJzKGNmZ1snZGF0YSddWydyb290J10sIGNmZ1snZGF0YSddWydzcGxpdHNfZGlyJ10sDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY2ZnWydkYXRhJ11bJ2JhdGNoX3NpemUnXSwgY2ZnWydkYXRhJ11bJ251bV93b3JrZXJzJ10sIDQyLCBUcnVlLCBUcnVlKQ0KICAgIHZhbCA9IGV2YWx1YXRlKG1vZGVsLCBsb2FkZXJzWyd2YWwnXSwgdG9yY2gubm4uQ3Jvc3NFbnRyb3B5TG9zcygpLCBkZXZpY2UpDQogICAgYXNzZXJ0IGFicyh2YWxbJ2FjYyddLW1ldHJpY3NbJ2Jlc3RfdmFsX2FjYyddKSA8IDFlLTYNCiAgICBhc3NlcnQgYWJzKHZhbFsnbG9zcyddLW1ldHJpY3NbJ2Jlc3RfdmFsX2xvc3MnXSkgPCAxZS02DQogICAgcmVzdWx0ID0geydzdGF0dXMnOiAnUEFTUycsICdzdHJpY3RfbG9hZCc6IHN0cihsb2FkZWQpLCAnYmFja2JvbmVfdGVuc29yX2tleXNfY2hlY2tlZCc6IGxlbihiYWNrYm9uZV9rZXlzKSwNCiAgICAgICAgICAgICAgJ2FsbF9iYWNrYm9uZV90ZW5zb3JzX2JpdHdpc2VfZXF1YWxfSW1hZ2VOZXQxS19WMSc6IFRydWUsICdoZWFkX3dlaWdodF9zaGFwZSc6IFszNywgNzY4XSwNCiAgICAgICAgICAgICAgJ3BhcmFtZXRlcnMnOiBjb3VudF9wYXJhbWV0ZXJzKG1vZGVsKSwgJ3ZhbGlkYXRpb25fcmVjb21wdXRlZCc6IHZhbCwNCiAgICAgICAgICAgICAgJ3ZhbGlkYXRpb25fbWF0Y2hlc19yZWNvcmRlZF9iZXN0JzogVHJ1ZSwgJ3ZhbGlkYXRpb25faW1hZ2VzJzogNzM2LA0KICAgICAgICAgICAgICAnZGV2aWNlJzogdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCksICd0ZXN0X2V2YWx1YXRpb24nOiAnTk9UX1JVTicsDQogICAgICAgICAgICAgICdjaGVja3BvaW50X3NoYTI1Nl9mcm9tX3ZlcmlmaWVkX3J1bic6IG1hbmlmZXN0WydjaGVja3BvaW50X3NoYTI1NiddfQ0KICAgIFBhdGgob3V0cHV0KS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9MikrJ1xuJywgZW5jb2Rpbmc9J3V0Zi04JykNCiAgICBwcmludCgnRTJfQ0hFQ0tQT0lOVF9BVURJVCcsIGpzb24uZHVtcHMocmVzdWx0KSwgZmx1c2g9VHJ1ZSkNCiAgICByZXR1cm4gcmVzdWx0DQoNCg0KaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoNCiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tcnVuJywgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLW91dHB1dCcsIHJlcXVpcmVkPVRydWUpDQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkNCiAgICBhdWRpdChhcmdzLnJ1biwgYXJncy5vdXRwdXQpDQo=', 'src/models/ky_independent_vit.py': 'IiIiSW5kZXBlbmRlbnQgVmlUIGFyY2hpdGVjdHVyZSwgaW1wbGVtZW50ZWQgYnkgTmd1eWVuIE5nb2MgS3kgdG8gc3VwcG9ydCBUcmluaC4KClVzZXMgUHlUb3JjaCBwcmltaXRpdmVzLCBub3QgdG9yY2h2aXNpb24gVmlUIGludGVybmFsbHkuIEFyY2hpdGVjdHVyZSByZWZlcmVuY2U6CkRvc292aXRza2l5IGV0IGFsLiBodHRwczovL2FyeGl2Lm9yZy9hYnMvMjAxMC4xMTkyOSA7IHRvcmNodmlzaW9uIHNvdXJjZToKaHR0cHM6Ly9kb2NzLnB5dG9yY2gub3JnL3Zpc2lvbi9zdGFibGUvX21vZHVsZXMvdG9yY2h2aXNpb24vbW9kZWxzL3Zpc2lvbl90cmFuc2Zvcm1lci5odG1sCk5vIHJlcGxhY2VtZW50IG9mIHRoZSBncm91cCdzIGZhY3Rvcnkgb3IgdHJhaW5pbmcgbW9kZWxzLiBXZWlnaHQgbWFwcGluZyBpcyBleHBsaWNpdC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KCgpjbGFzcyBTZWxmQXR0ZW50aW9uKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgZGltOiBpbnQsIGhlYWRzOiBpbnQpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGlmIGRpbSAlIGhlYWRzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFbWJlZGRpbmcgZGltZW5zaW9uIG11c3QgYmUgZGl2aXNpYmxlIGJ5IGhlYWRzIikKICAgICAgICBzZWxmLmhlYWRzID0gaGVhZHMKICAgICAgICBzZWxmLnFrdiA9IG5uLkxpbmVhcihkaW0sIDMgKiBkaW0pCiAgICAgICAgc2VsZi5wcm9qZWN0aW9uID0gbm4uTGluZWFyKGRpbSwgZGltKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIGJhdGNoLCB0b2tlbnMsIGRpbSA9IHguc2hhcGUKICAgICAgICBxa3YgPSBzZWxmLnFrdih4KS5yZXNoYXBlKGJhdGNoLCB0b2tlbnMsIDMsIHNlbGYuaGVhZHMsIGRpbSAvLyBzZWxmLmhlYWRzKQogICAgICAgIHEsIGssIHYgPSBxa3YucGVybXV0ZSgyLCAwLCAzLCAxLCA0KS51bmJpbmQoMCkKICAgICAgICBhdHRlbnRpb24gPSAoKHEgQCBrLnRyYW5zcG9zZSgtMiwgLTEpKSAvIChkaW0gLy8gc2VsZi5oZWFkcykgKiogMC41KS5zb2Z0bWF4KC0xKQogICAgICAgIHJldHVybiBzZWxmLnByb2plY3Rpb24oKGF0dGVudGlvbiBAIHYpLnRyYW5zcG9zZSgxLCAyKS5yZXNoYXBlKGJhdGNoLCB0b2tlbnMsIGRpbSkpCgoKY2xhc3MgRW5jb2RlckJsb2NrKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgZGltOiBpbnQsIGhlYWRzOiBpbnQsIG1scF9kaW06IGludCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5ub3JtMSA9IG5uLkxheWVyTm9ybShkaW0sIGVwcz0xZS02KQogICAgICAgIHNlbGYuYXR0ZW50aW9uID0gU2VsZkF0dGVudGlvbihkaW0sIGhlYWRzKQogICAgICAgIHNlbGYubm9ybTIgPSBubi5MYXllck5vcm0oZGltLCBlcHM9MWUtNikKICAgICAgICBzZWxmLm1scCA9IG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKGRpbSwgbWxwX2RpbSksIG5uLkdFTFUoKSwgbm4uTGluZWFyKG1scF9kaW0sIGRpbSkpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgeCA9IHggKyBzZWxmLmF0dGVudGlvbihzZWxmLm5vcm0xKHgpKQogICAgICAgIHJldHVybiB4ICsgc2VsZi5tbHAoc2VsZi5ub3JtMih4KSkKCgpjbGFzcyBJbmRlcGVuZGVudFZpVChubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG51bV9jbGFzc2VzPTM3LCBpbWFnZV9zaXplPTIyNCwgcGF0Y2hfc2l6ZT0xNiwgZGltPTc2OCwgZGVwdGg9MTIsIGhlYWRzPTEyLCBtbHBfZGltPTMwNzIpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGlmIGltYWdlX3NpemUgJSBwYXRjaF9zaXplOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJJbWFnZSBzaXplIG11c3QgYmUgZGl2aXNpYmxlIGJ5IHBhdGNoIHNpemUiKQogICAgICAgIHNlbGYuaW1hZ2Vfc2l6ZSA9IGltYWdlX3NpemUKICAgICAgICBzZWxmLnBhdGNoX2VtYmVkZGluZyA9IG5uLkNvbnYyZCgzLCBkaW0sIHBhdGNoX3NpemUsIHN0cmlkZT1wYXRjaF9zaXplKQogICAgICAgIHNlbGYuY2xhc3NfdG9rZW4gPSBubi5QYXJhbWV0ZXIodG9yY2guemVyb3MoMSwgMSwgZGltKSkKICAgICAgICBzZWxmLnBvc2l0aW9uX2VtYmVkZGluZyA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcygxLCAoaW1hZ2Vfc2l6ZSAvLyBwYXRjaF9zaXplKSAqKiAyICsgMSwgZGltKSkKICAgICAgICBzZWxmLmJsb2NrcyA9IG5uLk1vZHVsZUxpc3QoW0VuY29kZXJCbG9jayhkaW0sIGhlYWRzLCBtbHBfZGltKSBmb3IgXyBpbiByYW5nZShkZXB0aCldKQogICAgICAgIHNlbGYubm9ybSA9IG5uLkxheWVyTm9ybShkaW0sIGVwcz0xZS02KQogICAgICAgIHNlbGYuaGVhZCA9IG5uLkxpbmVhcihkaW0sIG51bV9jbGFzc2VzKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGltYWdlKToKICAgICAgICBpZiBpbWFnZS5uZGltICE9IDQgb3IgaW1hZ2Uuc2hhcGVbMTpdICE9ICgzLCBzZWxmLmltYWdlX3NpemUsIHNlbGYuaW1hZ2Vfc2l6ZSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4cGVjdGVkIE5DSFcgUkdCIGltYWdlcyBhdCBjb25maWd1cmVkIGltYWdlIHNpemUiKQogICAgICAgIHRva2VucyA9IHNlbGYucGF0Y2hfZW1iZWRkaW5nKGltYWdlKS5mbGF0dGVuKDIpLnRyYW5zcG9zZSgxLCAyKQogICAgICAgIHRva2VucyA9IHRvcmNoLmNhdChbc2VsZi5jbGFzc190b2tlbi5leHBhbmQoaW1hZ2Uuc2hhcGVbMF0sIC0xLCAtMSksIHRva2Vuc10sIGRpbT0xKQogICAgICAgIHRva2VucyA9IHRva2VucyArIHNlbGYucG9zaXRpb25fZW1iZWRkaW5nCiAgICAgICAgZm9yIGJsb2NrIGluIHNlbGYuYmxvY2tzOgogICAgICAgICAgICB0b2tlbnMgPSBibG9jayh0b2tlbnMpCiAgICAgICAgcmV0dXJuIHNlbGYuaGVhZChzZWxmLm5vcm0odG9rZW5zKVs6LCAwXSkKCiAgICBkZWYgbG9hZF90b3JjaHZpc2lvbl9zdGF0ZShzZWxmLCBzdGF0ZSk6CiAgICAgICAgIiIiTWFwIGFsbCBsZWFybmVkIHRlbnNvcnM7IHJlamVjdCB1bmNvbnN1bWVkIG9yIG1pc3Npbmcgc291cmNlIHdlaWdodHMuIiIiCiAgICAgICAgbWFwcGluZyA9IHsKICAgICAgICAgICAgInBhdGNoX2VtYmVkZGluZy53ZWlnaHQiOiAiY29udl9wcm9qLndlaWdodCIsCiAgICAgICAgICAgICJwYXRjaF9lbWJlZGRpbmcuYmlhcyI6ICJjb252X3Byb2ouYmlhcyIsCiAgICAgICAgICAgICJjbGFzc190b2tlbiI6ICJjbGFzc190b2tlbiIsCiAgICAgICAgICAgICJwb3NpdGlvbl9lbWJlZGRpbmciOiAiZW5jb2Rlci5wb3NfZW1iZWRkaW5nIiwKICAgICAgICAgICAgIm5vcm0ud2VpZ2h0IjogImVuY29kZXIubG4ud2VpZ2h0IiwKICAgICAgICAgICAgIm5vcm0uYmlhcyI6ICJlbmNvZGVyLmxuLmJpYXMiLAogICAgICAgICAgICAiaGVhZC53ZWlnaHQiOiAiaGVhZHMuaGVhZC53ZWlnaHQiLAogICAgICAgICAgICAiaGVhZC5iaWFzIjogImhlYWRzLmhlYWQuYmlhcyIsCiAgICAgICAgfQogICAgICAgIGZvciBpbmRleCBpbiByYW5nZShsZW4oc2VsZi5ibG9ja3MpKToKICAgICAgICAgICAgb3duLCB0diA9IGYiYmxvY2tzLntpbmRleH0uIiwgZiJlbmNvZGVyLmxheWVycy5lbmNvZGVyX2xheWVyX3tpbmRleH0uIgogICAgICAgICAgICBmb3IgdGFyZ2V0LCBzb3VyY2UgaW4gWwogICAgICAgICAgICAgICAgKCJub3JtMSIsICJsbl8xIiksCiAgICAgICAgICAgICAgICAoIm5vcm0yIiwgImxuXzIiKSwKICAgICAgICAgICAgICAgICgiYXR0ZW50aW9uLnFrdiIsICJzZWxmX2F0dGVudGlvbi5pbl9wcm9qIiksCiAgICAgICAgICAgICAgICAoImF0dGVudGlvbi5wcm9qZWN0aW9uIiwgInNlbGZfYXR0ZW50aW9uLm91dF9wcm9qIiksCiAgICAgICAgICAgICAgICAoIm1scC4wIiwgIm1scC4wIiksCiAgICAgICAgICAgICAgICAoIm1scC4yIiwgIm1scC4zIiksCiAgICAgICAgICAgIF06CiAgICAgICAgICAgICAgICBmb3IgZmllbGQgaW4gWyJ3ZWlnaHQiLCAiYmlhcyJdOgogICAgICAgICAgICAgICAgICAgIHNvdXJjZV9uYW1lID0gc291cmNlICsgKCJfIiBpZiBzb3VyY2UuZW5kc3dpdGgoImluX3Byb2oiKSBlbHNlICIuIikgKyBmaWVsZAogICAgICAgICAgICAgICAgICAgIG1hcHBpbmdbb3duICsgdGFyZ2V0ICsgIi4iICsgZmllbGRdID0gdHYgKyBzb3VyY2VfbmFtZQogICAgICAgIGlmIHNldChzdGF0ZSkgIT0gc2V0KG1hcHBpbmcudmFsdWVzKCkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTb3VyY2UgY2hlY2twb2ludCBrZXlzIGRpZmZlciBmcm9tIGV4cGVjdGVkIHRvcmNodmlzaW9uIGFyY2hpdGVjdHVyZSIpCiAgICAgICAgcmV0dXJuIHNlbGYubG9hZF9zdGF0ZV9kaWN0KHt0YXJnZXQ6IHN0YXRlW3NvdXJjZV0gZm9yIHRhcmdldCwgc291cmNlIGluIG1hcHBpbmcuaXRlbXMoKX0sIHN0cmljdD1UcnVlKQo=', 'src/ky_verify_vit.py': 'IiIiQ29tcGFyZSBLeSdzIGluZGVwZW5kZW50IFZpVCB0byB0b3JjaHZpc2lvbiB3aXRoIGlkZW50aWNhbCwgbm9uemVybyB0cmFpbmVkIHdlaWdodHMuCgpEZWNsYXJlZCB0b2xlcmFuY2VzIGJlZm9yZSBleGVjdXRpb246IGZsb2F0MzIgQ1BVIGF0b2w9MWUtNSwgcnRvbD0xZS00LgpGdXNlZCBhbmQgZXhwbGljaXQgYXR0ZW50aW9uIGNhbiBkaWZmZXIgYnkgZmxvYXRpbmctcG9pbnQgcm91bmRpbmcuIFRvcDEgbXVzdCBhZ3JlZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgdG9yY2gKZnJvbSBQSUwgaW1wb3J0IEltYWdlLCBJbWFnZU9wcwpmcm9tIHRvcmNodmlzaW9uLm1vZGVscyBpbXBvcnQgVmlUX0JfMTZfV2VpZ2h0cywgdml0X2JfMTYKCmZyb20gc3JjLm1vZGVscy5reV9pbmRlcGVuZGVudF92aXQgaW1wb3J0IEluZGVwZW5kZW50VmlUCgoKZGVmIHZlcmlmeShjaGVja3BvaW50LCBpbWFnZV9kaXIsIG91dHB1dCk6CiAgICB0b3JjaC5zZXRfbnVtX3RocmVhZHMoMikKICAgIHRvcmNoLm1hbnVhbF9zZWVkKDQyKQogICAgc3RhdGUgPSB0b3JjaC5sb2FkKGNoZWNrcG9pbnQsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PVRydWUpCiAgICBpZiBub3QgaXNpbnN0YW5jZShzdGF0ZSwgZGljdCkgb3IgImhlYWRzLmhlYWQud2VpZ2h0IiBub3QgaW4gc3RhdGU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhwZWN0ZWQgb2ZmaWNpYWwgcmF3IHRvcmNodmlzaW9uIHN0YXRlX2RpY3QiKQogICAgaWYgdG9yY2guY291bnRfbm9uemVybyhzdGF0ZVsiaGVhZHMuaGVhZC53ZWlnaHQiXSkgPT0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJBIHplcm8gY2xhc3NpZmllciBtYWtlcyBsb2dpdHMgY29tcGFyaXNvbiB2YWN1b3VzIikKICAgIGNsYXNzZXMgPSBzdGF0ZVsiaGVhZHMuaGVhZC53ZWlnaHQiXS5zaGFwZVswXQogICAgcmVmZXJlbmNlID0gdml0X2JfMTYod2VpZ2h0cz1Ob25lKQogICAgcmVmZXJlbmNlLmhlYWRzLmhlYWQgPSB0b3JjaC5ubi5MaW5lYXIoNzY4LCBjbGFzc2VzKQogICAgcmVmZXJlbmNlLmxvYWRfc3RhdGVfZGljdChzdGF0ZSwgc3RyaWN0PVRydWUpCiAgICBjYW5kaWRhdGUgPSBJbmRlcGVuZGVudFZpVChudW1fY2xhc3Nlcz1jbGFzc2VzKQogICAgY2FuZGlkYXRlLmxvYWRfdG9yY2h2aXNpb25fc3RhdGUoc3RhdGUpCiAgICByZWZlcmVuY2UuZXZhbCgpCiAgICBjYW5kaWRhdGUuZXZhbCgpCiAgICB0cmFuc2Zvcm0gPSBWaVRfQl8xNl9XZWlnaHRzLklNQUdFTkVUMUtfVjEudHJhbnNmb3JtcygpCiAgICBpbnB1dHMgPSBbKCJyYW5kb21fc2VlZDQyIiwgdG9yY2gucmFuZG4oMSwgMywgMjI0LCAyMjQpKSwgKCJ6ZXJvcyIsIHRvcmNoLnplcm9zKDEsIDMsIDIyNCwgMjI0KSldCiAgICBmb3IgaW1hZ2VfaWQgaW4gWyJBYnlzc2luaWFuXzEwMCIsICJiZWFnbGVfMTAwIl06CiAgICAgICAgd2l0aCBJbWFnZS5vcGVuKFBhdGgoaW1hZ2VfZGlyKSAvIChpbWFnZV9pZCArICIuanBnIikpIGFzIGltYWdlOgogICAgICAgICAgICBpbnB1dHMuYXBwZW5kKChpbWFnZV9pZCwgdHJhbnNmb3JtKEltYWdlT3BzLmV4aWZfdHJhbnNwb3NlKGltYWdlKS5jb252ZXJ0KCJSR0IiKSkudW5zcXVlZXplKDApKSkKICAgIHJlY29yZHMgPSBbXQogICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOgogICAgICAgIGZvciBuYW1lLCB4IGluIGlucHV0czoKICAgICAgICAgICAgZXhwZWN0ZWQsIGFjdHVhbCA9IHJlZmVyZW5jZSh4KSwgY2FuZGlkYXRlKHgpCiAgICAgICAgICAgIGVycm9yID0gKGV4cGVjdGVkIC0gYWN0dWFsKS5hYnMoKQogICAgICAgICAgICByZWNvcmQgPSB7CiAgICAgICAgICAgICAgICAiaW5wdXQiOiBuYW1lLAogICAgICAgICAgICAgICAgInNoYXBlIjogbGlzdChhY3R1YWwuc2hhcGUpLAogICAgICAgICAgICAgICAgIm1heF9hYnNfZXJyb3IiOiBmbG9hdChlcnJvci5tYXgoKSksCiAgICAgICAgICAgICAgICAibWVhbl9hYnNfZXJyb3IiOiBmbG9hdChlcnJvci5tZWFuKCkpLAogICAgICAgICAgICAgICAgImFsbGNsb3NlIjogdG9yY2guYWxsY2xvc2UoYWN0dWFsLCBleHBlY3RlZCwgYXRvbD0xZS01LCBydG9sPTFlLTQpLAogICAgICAgICAgICAgICAgInJlZmVyZW5jZV90b3AxIjogZXhwZWN0ZWQuYXJnbWF4KC0xKS50b2xpc3QoKSwKICAgICAgICAgICAgICAgICJpbmRlcGVuZGVudF90b3AxIjogYWN0dWFsLmFyZ21heCgtMSkudG9saXN0KCksCiAgICAgICAgICAgICAgICAicmVmZXJlbmNlX2xvZ2l0X21heF9hYnMiOiBmbG9hdChleHBlY3RlZC5hYnMoKS5tYXgoKSksCiAgICAgICAgICAgIH0KICAgICAgICAgICAgcmVjb3Jkcy5hcHBlbmQocmVjb3JkKQogICAgcmVzdWx0ID0gewogICAgICAgICJhdXRob3IiOiAiTmd1eWVuIE5nb2MgS3kgc3VwcG9ydGluZyBUcmluaCIsCiAgICAgICAgImFyY2hpdGVjdHVyZSI6ICJWaVQtQi8xNiwgMTIgYmxvY2tzLCAxMiBoZWFkcywgZGltNzY4LCBNTFAzMDcyIiwKICAgICAgICAiaW1wbGVtZW50YXRpb24iOiAic3JjL21vZGVscy9reV9pbmRlcGVuZGVudF92aXQucHk7IG5vIHRvcmNodmlzaW9uIG1vZGVsIGluc2lkZSIsCiAgICAgICAgImRldmljZSI6ICJjcHUiLAogICAgICAgICJkdHlwZSI6ICJmbG9hdDMyIiwKICAgICAgICAiZXZhbF9tb2RlIjogVHJ1ZSwKICAgICAgICAiZHJvcG91dCI6IDAsCiAgICAgICAgImF0b2wiOiAxZS01LAogICAgICAgICJydG9sIjogMWUtNCwKICAgICAgICAid2VpZ2h0X3NvdXJjZSI6IHN0cihjaGVja3BvaW50KSwKICAgICAgICAiY2hlY2twb2ludF9zaGEyNTYiOiBoYXNobGliLnNoYTI1NihQYXRoKGNoZWNrcG9pbnQpLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCksCiAgICAgICAgImltcGxlbWVudGF0aW9uX3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KFBhdGgoInNyYy9tb2RlbHMva3lfaW5kZXBlbmRlbnRfdml0LnB5IikucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAiY2FzZXMiOiByZWNvcmRzLAogICAgICAgICJzdGF0dXMiOiAiUEFTUyIKICAgICAgICBpZiBhbGwoclsiYWxsY2xvc2UiXSBhbmQgclsicmVmZXJlbmNlX3RvcDEiXSA9PSByWyJpbmRlcGVuZGVudF90b3AxIl0gZm9yIHIgaW4gcmVjb3JkcykKICAgICAgICBlbHNlICJGQUlMRUQiLAogICAgfQogICAgUGF0aChvdXRwdXQpLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBQYXRoKG91dHB1dCkud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlc3VsdCwgaW5kZW50PTIpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHByaW50KCJLWV9WSVRfTE9HSVRTX1JFU1VMVCIsIGpzb24uZHVtcHMocmVzdWx0KSwgZmx1c2g9VHJ1ZSkKICAgIGlmIHJlc3VsdFsic3RhdHVzIl0gIT0gIlBBU1MiOgogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKCJJbmRlcGVuZGVudCBWaVQgZG9lcyBub3QgbWVldCBkZWNsYXJlZCB0b2xlcmFuY2VzIikKICAgIHJldHVybiByZXN1bHQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY2hlY2twb2ludCIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWltYWdlLWRpciIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dCIsIHJlcXVpcmVkPVRydWUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgdmVyaWZ5KGFyZ3MuY2hlY2twb2ludCwgYXJncy5pbWFnZV9kaXIsIGFyZ3Mub3V0cHV0KQo=', 'tests/test_ky_e2_independent_vit.py': 'IiIiVmVyaWZ5IGluZGVwZW5kZW50IGF0dGVudGlvbi9WaVQgYW5kIGxpbmVhci1wcm9iZSBvcHRpbWl6ZXIgaXNvbGF0aW9uLiIiIgoKaW1wb3J0IHB5dGVzdAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaHZpc2lvbi5tb2RlbHMudmlzaW9uX3RyYW5zZm9ybWVyIGltcG9ydCBWaXNpb25UcmFuc2Zvcm1lcgoKZnJvbSBzcmMubW9kZWxzLmZhY3RvcnkgaW1wb3J0IGJ1aWxkX21vZGVsCmZyb20gc3JjLm1vZGVscy5reV9pbmRlcGVuZGVudF92aXQgaW1wb3J0IEluZGVwZW5kZW50VmlUCmZyb20gc3JjLm9wdGltIGltcG9ydCBidWlsZF9vcHRpbWl6ZXIKZnJvbSBzcmMudXRpbHMgaW1wb3J0IGxvYWRfY29uZmlnCgoKZGVmIHRlc3RfaW5kZXBlbmRlbnRfbG9naXRzX25vbnplcm9faGVhZF9hbmRfd2VpZ2h0X21hcHBpbmcoKToKICAgIHRvcmNoLm1hbnVhbF9zZWVkKDQyKQogICAgcmVmZXJlbmNlID0gVmlzaW9uVHJhbnNmb3JtZXIoMzIsIDE2LCAyLCAzLCAxMiwgMjQsIG51bV9jbGFzc2VzPTUpLmV2YWwoKQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgcmVmZXJlbmNlLmhlYWRzLmhlYWQud2VpZ2h0Lm5vcm1hbF8oc3RkPTAuMSkKICAgICAgICByZWZlcmVuY2UuaGVhZHMuaGVhZC5iaWFzLm5vcm1hbF8oc3RkPTAuMSkKICAgIGNhbmRpZGF0ZSA9IEluZGVwZW5kZW50VmlUKDUsIDMyLCAxNiwgMTIsIDIsIDMsIDI0KS5ldmFsKCkKICAgIGNhbmRpZGF0ZS5sb2FkX3RvcmNodmlzaW9uX3N0YXRlKHJlZmVyZW5jZS5zdGF0ZV9kaWN0KCkpCiAgICB4ID0gdG9yY2gucmFuZG4oMiwgMywgMzIsIDMyKQogICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOgogICAgICAgIGFjdHVhbCwgZXhwZWN0ZWQgPSBjYW5kaWRhdGUoeCksIHJlZmVyZW5jZSh4KQogICAgYXNzZXJ0IGFjdHVhbC5hYnMoKS5tYXgoKSA+IDAuMDEKICAgIHRvcmNoLnRlc3RpbmcuYXNzZXJ0X2Nsb3NlKGFjdHVhbCwgZXhwZWN0ZWQsIGF0b2w9MWUtNSwgcnRvbD0xZS00KQogICAgYXNzZXJ0IHRvcmNoLmVxdWFsKGFjdHVhbC5hcmdtYXgoLTEpLCBleHBlY3RlZC5hcmdtYXgoLTEpKQogICAgYmFkID0gZGljdChyZWZlcmVuY2Uuc3RhdGVfZGljdCgpKQogICAgYmFkLnBvcCgiY2xhc3NfdG9rZW4iKQogICAgd2l0aCBweXRlc3QucmFpc2VzKFZhbHVlRXJyb3IsIG1hdGNoPSJjaGVja3BvaW50IGtleXMiKToKICAgICAgICBjYW5kaWRhdGUubG9hZF90b3JjaHZpc2lvbl9zdGF0ZShiYWQpCgoKZGVmIHRlc3RfZTJfZnJlZXplX2JhY2t3YXJkX2FuZF9vcHRpbWl6ZXJfb25seV91cGRhdGVzX2hlYWQoKToKICAgIGNmZyA9IGxvYWRfY29uZmlnKCJjb25maWdzL2UyX3ZpdF9saW5lYXJfcHJvYmUueWFtbCIpCiAgICBhc3NlcnQgY2ZnWyJvcHRpbWl6ZXIiXVsibHIiXSA9PSAwLjAwMyBhbmQgY2ZnWyJkYXRhIl1bImJhdGNoX3NpemUiXSA9PSA2NAogICAgYXNzZXJ0IGNmZ1sidHJhaW4iXVsiZXBvY2hzIl0gPT0gMTAgYW5kIGNmZ1sidHJhaW4iXVsic2VlZCJdID09IDQyCiAgICBtb2RlbF9jZmcgPSBkaWN0KGNmZ1sibW9kZWwiXSwgcHJldHJhaW5lZD1GYWxzZSkKICAgIG1vZGVsID0gYnVpbGRfbW9kZWwobW9kZWxfY2ZnKQogICAgdHJhaW5hYmxlID0ge24gZm9yIG4sIHAgaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpIGlmIHAucmVxdWlyZXNfZ3JhZH0KICAgIGFzc2VydCB0cmFpbmFibGUgPT0geyJoZWFkcy5oZWFkLndlaWdodCIsICJoZWFkcy5oZWFkLmJpYXMifQogICAgYXNzZXJ0IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpIGlmIHAucmVxdWlyZXNfZ3JhZCkgPT0gMjg0NTMKICAgIG9wdGltaXplciA9IGJ1aWxkX29wdGltaXplcihtb2RlbCwgY2ZnWyJvcHRpbWl6ZXIiXSkKICAgIGFzc2VydCB7aWQocCkgZm9yIGcgaW4gb3B0aW1pemVyLnBhcmFtX2dyb3VwcyBmb3IgcCBpbiBnWyJwYXJhbXMiXX0gPT0gewogICAgICAgIGlkKHApIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSBpZiBwLnJlcXVpcmVzX2dyYWQKICAgIH0KICAgIGJlZm9yZSA9IHtuOiBwLmRldGFjaCgpLmNsb25lKCkgZm9yIG4sIHAgaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpfQogICAgbG9naXRzID0gbW9kZWwodG9yY2gucmFuZG4oMSwgMywgMjI0LCAyMjQpKQogICAgbG9zcyA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwuY3Jvc3NfZW50cm9weShsb2dpdHMsIHRvcmNoLnRlbnNvcihbM10pKQogICAgYXNzZXJ0IHRvcmNoLmlzZmluaXRlKGxvc3MpCiAgICBsb3NzLmJhY2t3YXJkKCkKICAgIGFzc2VydCBhbGwocC5ncmFkIGlzIE5vbmUgZm9yIG4sIHAgaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpIGlmIG4gbm90IGluIHRyYWluYWJsZSkKICAgIG9wdGltaXplci5zdGVwKCkKICAgIGNoYW5nZWQgPSB7biBmb3IgbiwgcCBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCkgaWYgbm90IHRvcmNoLmVxdWFsKHAsIGJlZm9yZVtuXSl9CiAgICBhc3NlcnQgY2hhbmdlZCBhbmQgY2hhbmdlZCA8PSB0cmFpbmFibGUKCgpkZWYgdGVzdF9mdWxsX3ZpdF9iMTZfbG9naXRzX3dpdGhfaWRlbnRpY2FsX25vbnplcm9fd2VpZ2h0cygpOgogICAgdG9yY2gubWFudWFsX3NlZWQoNykKICAgIHJlZmVyZW5jZSA9IGJ1aWxkX21vZGVsKHsibmFtZSI6ICJ2aXRfYl8xNiIsICJudW1fY2xhc3NlcyI6IDM3LCAicHJldHJhaW5lZCI6IEZhbHNlLCAic3RyYXRlZ3kiOiAiZnVsbCJ9KQogICAgcmVmZXJlbmNlLmV2YWwoKQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgcmVmZXJlbmNlLmhlYWRzLmhlYWQud2VpZ2h0Lm5vcm1hbF8oc3RkPTAuMDIpCiAgICAgICAgcmVmZXJlbmNlLmhlYWRzLmhlYWQuYmlhcy5ub3JtYWxfKHN0ZD0wLjAxKQogICAgY2FuZGlkYXRlID0gSW5kZXBlbmRlbnRWaVQoKS5ldmFsKCkKICAgIGNhbmRpZGF0ZS5sb2FkX3RvcmNodmlzaW9uX3N0YXRlKHJlZmVyZW5jZS5zdGF0ZV9kaWN0KCkpCiAgICBhc3NlcnQgbGVuKGNhbmRpZGF0ZS5ibG9ja3MpID09IDEyIGFuZCBjYW5kaWRhdGUuYmxvY2tzWzBdLmF0dGVudGlvbi5oZWFkcyA9PSAxMgogICAgYXNzZXJ0IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gY2FuZGlkYXRlLnBhcmFtZXRlcnMoKSkgPT0gODU4MjcxMDkKICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICBmb3IgeCBpbiBbdG9yY2gucmFuZG4oMSwgMywgMjI0LCAyMjQpLCB0b3JjaC56ZXJvcygxLCAzLCAyMjQsIDIyNCldOgogICAgICAgICAgICBleHBlY3RlZCwgYWN0dWFsID0gcmVmZXJlbmNlKHgpLCBjYW5kaWRhdGUoeCkKICAgICAgICAgICAgYXNzZXJ0IGFjdHVhbC5zaGFwZSA9PSAoMSwgMzcpIGFuZCBleHBlY3RlZC5hYnMoKS5tYXgoKSA+IDAuMDEKICAgICAgICAgICAgZXJyb3IgPSAoYWN0dWFsIC0gZXhwZWN0ZWQpLmFicygpCiAgICAgICAgICAgIHByaW50KCJGVUxMX1ZJVF9MT0dJVFMiLCBmbG9hdChlcnJvci5tYXgoKSksIGZsb2F0KGVycm9yLm1lYW4oKSkpCiAgICAgICAgICAgIHRvcmNoLnRlc3RpbmcuYXNzZXJ0X2Nsb3NlKGFjdHVhbCwgZXhwZWN0ZWQsIGF0b2w9MWUtNSwgcnRvbD0xZS00KQogICAgICAgICAgICBhc3NlcnQgdG9yY2guZXF1YWwoYWN0dWFsLmFyZ21heCgtMSksIGV4cGVjdGVkLmFyZ21heCgtMSkpDQo='}
for name,encoded in FILES.items():
    p=Path(name);data=base64.b64decode(encoded)
    if p.exists():
        assert p.read_bytes()==data, 'Preserve different existing scoped file: '+name
    else:
        p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(data)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt','pytest'],check=True)
os.environ['MPLCONFIGDIR']='/content/ky-e2-cache/mpl'
os.environ['TORCH_HOME']='/content/ky-e2-cache/torch'
subprocess.run([sys.executable,'-m','pytest','tests/test_ky_e2_independent_vit.py','-q','-p','no:cacheprovider'],check=True)
from src.data import build_dataloaders
from torchvision.datasets import OxfordIIITPet
OxfordIIITPet('/content/data',split='trainval',download=True)
loaders=build_dataloaders('/content/data','data/splits',64,2,42,True,True)
assert {k:len(v.dataset) for k,v in loaders.items()}=={'train':2944,'val':736,'test':3669}
print('E2_PREFLIGHT_PASS',flush=True)

assert torch.cuda.is_available(), 'BLOCKED: E2 preflight software checks completed but no GPU for real training'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json,hashlib,subprocess,sys
from pathlib import Path
assert torch.cuda.is_available() and Path('/content/drive/MyDrive').is_dir()
from src.utils import load_config
store=Path('/content/drive/MyDrive/vit-oxford-pets-ky/e2-support')
completed=[]
for manifest_path in sorted(store.glob('*/run_metadata.json')):
    record=json.loads(manifest_path.read_text())
    directory=manifest_path.parent/'results/e2_vit_linear_probe/seed42'
    checkpoint=manifest_path.parent/'checkpoints/e2_vit_linear_probe/seed42/best.pt'
    if record.get('status')!='DONE' or record.get('source_commit')!='3359b2bdd2a081faa20f9ff83a93c2ccbac12071':
        continue
    if record.get('helper_sha256')!=hashlib.sha256(Path('src/ky_e2_runner.py').read_bytes()).hexdigest():
        continue
    if load_config(directory/'config.yaml')!=load_config('configs/e2_vit_linear_probe.yaml'):
        continue
    if checkpoint.is_file() and hashlib.sha256(checkpoint.read_bytes()).hexdigest()==record.get('checkpoint_sha256'):
        completed.append(manifest_path.parent)
if completed:
    run=completed[-1]
    print('Reuse completed verified E2; no retraining:',run)
else:
    subprocess.run([sys.executable,'-u','-m','src.ky_e2_runner'],check=True)
    complete=sorted(store.glob('*/run_metadata.json'))
    run=next(p.parent for p in reversed(complete) if json.loads(p.read_text()).get('status')=='DONE')


In [ ]:
import json
from pathlib import Path
from src.ky_e2_checkpoint_audit import audit
store=Path('/content/drive/MyDrive/vit-oxford-pets-ky/e2-support')
assert 'run' in globals(),'Run guarded training/reuse cell first'
assert json.loads((run/'run_metadata.json').read_text())['status']=='DONE'
print('E2_METRICS', (run/'results/e2_vit_linear_probe/seed42/metrics.json').read_text())
audit(run,run/'checkpoint_verification.json')
from IPython.display import Image,display
display(Image(filename=str(run/'e2_curves.png')))
